## Create QA Pairs

Run after the figures exist (`single/create_figures_batch.py` or
`create_figures_batch.py`), which writes `imgs/` + `jsons/` into a run directory.
This reads each figure's json and writes a `*_qa.json` alongside it containing
the generated question/answer pairs under a `VQA` key.

**Scope:** this copy generates only

1. **full-figure questions** - panel count, plotting style, colormap, aspect
   ratio, titles, axis labels, tick labels, which plot types appear
2. **contour-panel questions** - image-vs-lines, min/max/median/mean on
   x/y/color, distribution of color and x/y

plus the plot-type-agnostic error-bar question. Histogram, scatter, line and
cross-panel questions are not generated - see `models/utils/README.md` for why
(cross-panel in particular *looks* figure-level but is scatter/line/histogram
only).

**Note on sky panels:** `image of the sky` panels get only the full-figure and
error-bar questions - there is no sky-specific question module yet.


In [1]:
# Run directory written by the figure generator: expects <save_dir_in>/jsons/
save_dir_in = "~/Dropbox/WASP2026/data/full_dataset/"

# ---- released VQA set (safe to share) ----
save_dir_out_json = "~/astro_sky_image_vqa/VQA_full/qa_jsons/"   # <vqa_id>_qa.json
save_dir_out_imgs = "~/astro_sky_image_vqa/VQA_full/imgs/"       # <vqa_id>.jpeg

# ---- PRIVATE: keep out of the released tree ----
# The generator's own filenames encode the answer to the Level 3 question --
# Picture_1xxxxx is a synthetic (gmm) sky, Picture_2xxxxx is a real cutout.
# The models never see filenames (payloads are base64 pixels), but humans,
# sorting tools and anything published would. So the released figures are
# re-issued under shuffled opaque ids and this table maps them back.
lookup_table_path = "~/Dropbox/WASP2026/data/VQA_private/vqa_lookup.json"
shuffle_seed = 20260913     # fixed so the build is reproducible

img_format = 'jpeg' 


In [2]:
import json
from glob import glob
import os
import sys
from copy import deepcopy
from importlib import reload
import numpy as np
from functools import partial

# This notebook lives at the repo root; the QA code lives in models/utils/.
# models/ has no __init__.py, so models.utils is picked up as a namespace
# package once the repo root is on sys.path.
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import models.utils.figure_level_qa_utils
import models.utils.contour_plot_qa_utils
import models.utils.sky_plot_qa_utils
import models.utils.qa_dispatch
reload(models.utils.figure_level_qa_utils)
reload(models.utils.contour_plot_qa_utils)
reload(models.utils.sky_plot_qa_utils)
reload(models.utils.qa_dispatch)

from models.utils.plot_qa_utils import init_qa_pairs
from models.utils.figure_level_qa_utils import figure_level_qa
from models.utils.misc_data_utils import NumpyEncoder
from models.utils.anonymize import (build_index_map, write_lookup_table,
                                    copy_image_as, resolve)
# dispatchers (composition layer, lifted out of this notebook into models/utils/)
from models.utils.qa_dispatch import (plot_level_contour_qa, plot_level_sky_qa,
                                      plot_level_general_qa, STATS, LINE_LIST)
from models.utils.sky_plot_qa_utils import SKY_LINE_LIST

stats = STATS
line_list = LINE_LIST


In [3]:
save_dir_in       = os.path.expanduser(save_dir_in)
save_dir_out_json = os.path.expanduser(save_dir_out_json)
save_dir_out_imgs = os.path.expanduser(save_dir_out_imgs)
lookup_table_path = os.path.expanduser(lookup_table_path)

for d in (save_dir_out_json, save_dir_out_imgs, os.path.dirname(lookup_table_path)):
    if not os.path.exists(d):
        os.makedirs(d, exist_ok=True)
        print('made:', d)

In [4]:
files_in = glob(save_dir_in + 'jsons/*.json')
files_in[:3]

['/Users/jnaiman/Downloads/tmp/test5_big/jsons/Picture_000571.json',
 '/Users/jnaiman/Downloads/tmp/test5_big/jsons/Picture_100613.json',
 '/Users/jnaiman/Downloads/tmp/test5_big/jsons/Picture_200646.json']

In [5]:
# get all plot types that have been used
plot_types = np.array([])
imod = 500
for ii,fi in enumerate(files_in):
    if ii%imod == 0: print('on', ii, 'of', len(files_in))
    with open(fi, 'r') as f:
        data = json.load(f)
        data = json.loads(data)
    for k, v in data.items():
        if 'plot' in k:
            plot_types = np.unique(np.concatenate([plot_types, [v['type']]]))
plot_types = plot_types.tolist()

plot_types

on 0 of 2001
on 500 of 2001
on 1000 of 2001
on 1500 of 2001
on 2000 of 2001


['contour', 'image of the sky']

In [6]:
# assign each figure a shuffled opaque id. Re-running does NOT reshuffle:
# ids already in the table are kept, only genuinely new figures get new ids.
source_stems = [os.path.basename(f).removesuffix('.json') for f in files_in]
id_map, lookup_table = build_index_map(source_stems,
                                       lookup_path=lookup_table_path,
                                       seed=shuffle_seed)

for ii,fi in enumerate(files_in):
    if ii%imod == 0:
        print('on', ii,'of',len(files_in))
    with open(fi, 'r') as f:
        data = json.load(f)
        data = json.loads(data)          # the figure jsons are double-encoded

    qa_pairs = init_qa_pairs()
    plot_nums = []
    for k, v in data.items():
        if 'plot' in k:
            plot_nums.append(int(k.split('plot')[-1]))

    # ---- (1) full-figure questions ----
    qa_pairs = figure_level_qa(data, deepcopy(qa_pairs), plot_types, verbose=False)

    # ---- per-panel questions ----
    for iplot in plot_nums:
        # leave off error bar gen
        # plot-type-agnostic (error bars), Level 2
        # qa_pairs = plot_level_general_qa(data, qa_pairs, iplot, verbose_qa=False)

        ptype = data['plot' + str(iplot)]['type']
        if ptype == 'contour':
            # ---- (2) contour-panel questions ----
            qa_pairs = plot_level_contour_qa(data, qa_pairs, iplot, stats,
                                             line_list=line_list, verbose_qa=False)
        elif ptype == 'image of the sky':
            # ---- (3) sky-panel questions ----
            # RA/DEC statistics come from the panel's WCS (xs/ys are pixel
            # indices, which do not match the RA/DEC printed on the axes).
            # The image-vs-lines question is off by default: the generator makes
            # it "image" ~99.8% of the time.
            qa_pairs = plot_level_sky_qa(data, qa_pairs, iplot, stats,
                                         line_list=SKY_LINE_LIST,
                                         ask_radec=False,
                                         ask_image_or_lines=False,
                                         verbose_qa=False)


    # update
    stem = os.path.basename(fi).removesuffix('.json')
    vqa_id = id_map[stem]

    # the figure, re-issued under its opaque id
    src_img = os.path.join(save_dir_in, 'imgs', stem + '.' + img_format)
    if copy_image_as(src_img, save_dir_out_imgs, vqa_id, ext=img_format) is None:
        print('[WARN] no image for', stem, '-- skipping')
        continue

    json_out = deepcopy(data)
    json_out['VQA'] = deepcopy(qa_pairs)
    # record the id inside the file too, so a stray qa json is still identifiable
    json_out['vqa_id'] = vqa_id
    dumped = json.dumps(json_out, cls=partial(NumpyEncoder, verbose=True))
    with open(os.path.join(save_dir_out_json, vqa_id + '_qa.json'), 'w') as f:
        json.dump(dumped, f)

write_lookup_table(lookup_table, lookup_table_path)
print('!!!!! DONE !!!!')
print('released :', save_dir_out_imgs, '+', save_dir_out_json)
print('PRIVATE  :', lookup_table_path, '-- do not ship this with the dataset')


index map: 2001 total (0 new this build), seed=20260913
on 0 of 2001
on 500 of 2001
on 1000 of 2001
on 1500 of 2001
on 2000 of 2001
wrote lookup table: /Users/jnaiman/Dropbox/WASP2026/VQA_private/vqa_lookup.json (2001 entries -- keep private)
!!!!! DONE !!!!
released : /Users/jnaiman/Dropbox/WASP2026/VQA/imgs/ + /Users/jnaiman/Dropbox/WASP2026/VQA/qa_jsons/
PRIVATE  : /Users/jnaiman/Dropbox/WASP2026/VQA_private/vqa_lookup.json -- do not ship this with the dataset


In [7]:
qa_pairs['Level 3']

{'Plot-level questions': {'distribution-image + list)': {'plot0': {'Q': 'You are a helpful assistant that can analyze images.  Please choose the distribution from the following list: [gaussian mixture model, real image of the sky]. What is the underlying distribution used to create the data in this figure? Please format the output as a json as {"distribution":""} for this figure, where the "distribution" value should be a string, calculated from the data values used to create the plot.',
    'A': {'distribution + list)': 'real image of the sky'},
    'note': 'sky panels are either a real SkyView cutout or a synthetic gaussian-mixture sky',
    'persona': 'You are a helpful assistant that can analyze images.',
    'context': ' Please choose the distribution from the following list: [gaussian mixture model, real image of the sky].',
    'question': 'What is the underlying distribution used to create the data in this figure?',
    'format': 'Please format the output as a json as {"distrib

In [8]:
# look at one
iPlot = 1

fi = save_dir_out_json + '/' + 'vqa_' + str(iPlot).zfill(6) + '_qa.json'
with open(fi, 'r') as f:
    data = json.load(f)
    data = json.loads(data)

In [9]:
data['VQA']

{'Level 1': {'Figure-level questions': {'plot style': {'Q': 'You are a helpful assistant that can analyze images. Assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure.',
    'A': {'plot style': 'classic'},
    'persona': 'You are a helpful assistant that can analyze images.',
    'context': 'Assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot".',
    'question': 'What is the plot style used in this figure?',
    'format': 'Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure.'},
   'colormap': {'Q': 'You are a helpful assistant that can analyze images. Assume this is a figure made with matplotlib in Python. Examples of matplotlib colormaps are "rainbow" or